# 12 — Sprint 4: análise comercial de uma nova transcrição

Este notebook integra os resultados do Challenge em um único fluxo: **Transcrição → Processamento → Modelos → Indicadores → Recomendação**. Ele processa uma transcrição por vez, preserva o texto original e exige revisão humana antes de qualquer ação comercial.

## 1. Contrato da análise

A saída contém produto principal, candidatos ranqueados, sentimento, risco de churn, oportunidade comercial, termos principais, recomendação e proveniência dos scores. Os modos `auto`, `full` e `fallback` são declarados no resultado.

## 2. Produto identificado e candidatos

O fallback local reutiliza a base TOTVS e os grupos de aliases do Challenge. O ranking BM25 dá mais peso a produto, título e keywords, consolida documentos por produto e mantém somente candidatos com fontes. O score é heurístico e relativo ao melhor candidato; não é probabilidade de compra.

## 3. Sentimento

No modo completo, o notebook carrega `pysentimiento/bertweet-pt-sentiment` sob demanda e agrega as probabilidades de chunks em `positivo`, `neutro`, `negativo` ou `misto`. No modo `auto`, a indisponibilidade do checkpoint aciona um fallback lexical com tratamento explícito de negação; no modo `full`, a ausência do modelo interrompe a execução.

## 4. Risco de churn

O modo completo usa `MoritzLaurer/multilingual-MiniLMv2-L6-mnli-xnli` para comparar hipóteses em português de risco `baixo`, `medio` e `alto`. Esse score zero-shot não é probabilidade calibrada de cancelamento. O fallback identifica sinais explícitos de cancelamento, não renovação, migração e concorrência, preservando a revisão humana obrigatória.

A oportunidade comercial usa o checkpoint local `data/processed/bertimbau_opportunity_best`, treinado na Sprint 3, e agrega as probabilidades dos chunks. Quando o artefato não está disponível, o modo `auto` aplica o fallback lexical de intenção, compra, dor e produto; ambos os caminhos produzem apenas um sinal para revisão humana.

In [ ]:
from __future__ import annotations

from collections import Counter
from functools import lru_cache
import json
import math
from pathlib import Path
import re
from typing import Any
import unicodedata

SUPPORTED_MODES = {"auto", "full", "fallback"}
SENTIMENT_MODEL_NAME = "pysentimiento/bertweet-pt-sentiment"
_SENTIMENT_ANALYZER = None
_SENTIMENT_LOAD_ERROR = None
CHURN_MODEL_NAME = "MoritzLaurer/multilingual-MiniLMv2-L6-mnli-xnli"
CHURN_HYPOTHESES = {
    "baixo": "baixo risco de cancelar ou trocar o fornecedor",
    "medio": "risco moderado de cancelar ou trocar o fornecedor",
    "alto": "alto risco de cancelar ou trocar o fornecedor",
}
_CHURN_CLASSIFIER = None
_CHURN_LOAD_ERROR = None
OPPORTUNITY_MODEL_PATH = "data/processed/bertimbau_opportunity_best"
_OPPORTUNITY_CLASSIFIER = None
_OPPORTUNITY_LOAD_ERROR = None
POSITIVE_SENTIMENT = {
    "gostei": 2.0, "otimo": 2.0, "excelente": 2.0,
    "satisfeito": 2.0, "satisfeita": 2.0, "satisfeitos": 2.0,
    "funciona bem": 2.0, "aprovado": 1.5, "bom": 1.0,
    "melhorou": 1.0, "interesse": 1.0,
}
NEGATIVE_SENTIMENT = {
    "insatisfeito": 2.0, "insatisfeita": 2.0, "insatisfeitos": 2.0,
    "problema": 1.0, "ruim": 2.0, "pessimo": 2.0,
    "cancelar": 2.0, "reclamacao": 1.5, "frustrado": 2.0,
    "nao funciona": 2.0, "falha": 1.0, "dificuldade": 1.0,
}
HIGH_CHURN_SIGNALS = {
    "cancelar": 3.0, "rescindir": 3.0, "encerrar o contrato": 3.0,
    "nao renovar": 3.0, "nao vamos renovar": 3.0,
    "trocar de fornecedor": 2.5, "migrar": 2.0,
    "concorrente": 1.5, "sair da totvs": 3.0,
}
MEDIUM_CHURN_SIGNALS = {
    "insatisfeito": 1.5, "insatisfeita": 1.5, "problema recorrente": 1.5,
    "problema": 1.0, "avaliando alternativas": 1.5,
    "reduzir o uso": 1.5, "preco alto": 1.0, "reclamacao": 1.0,
}
RETENTION_SIGNALS = {
    "renovar": 2.0, "continuar": 1.0, "satisfeito": 1.0,
    "satisfeita": 1.0, "expansao": 1.0,
}
OPPORTUNITY_INTENT_SIGNALS = {
    "precisamos", "preciso", "necessitamos", "queremos",
    "gostariamos", "buscamos", "procuramos", "avaliar",
    "avaliando", "interesse", "interessado", "interessados",
}
OPPORTUNITY_BUY_SIGNALS = {
    "proposta", "cotacao", "orcamento", "preco", "valor",
    "contratar", "adquirir", "comprar", "implantacao",
    "implantar", "implementar", "substituir", "prazo",
}
OPPORTUNITY_PAIN_SIGNALS = {
    "problema", "dificuldade", "gargalo", "retrabalho", "manual",
    "planilha", "integracao", "lentidao", "erro", "nao atende",
    "limitacao",
}
OPPORTUNITY_PRODUCT_SIGNALS = {
    "erp", "crm", "software", "sistema", "solucao", "plataforma",
    "protheus", "fluig", "datasul", "totvs rm", "hcm", "wms",
    "licenca", "modulo",
}
OPPORTUNITY_REFUSAL_SIGNALS = {
    "sem interesse", "nao temos interesse", "nao precisamos",
    "nao pretendemos", "nao vamos contratar", "projeto cancelado",
    "projeto descartado",
}
PORTUGUESE_STOPWORDS = {
    "a", "ao", "aos", "as", "com", "como", "da", "das",
    "de", "do", "dos", "e", "em", "entre", "essa", "esse",
    "esta", "este", "foi", "isso", "mais", "mas", "muito",
    "na", "nao", "nas", "no", "nos", "o", "os", "ou",
    "para", "pela", "pelo", "por", "que", "se", "sem", "ser",
    "sua", "sao", "tem", "um", "uma", "voce", "cliente",
    "reuniao", "atual", "cenario", "acompanhamento",
}


def _indicator(label: str) -> dict[str, Any]:
    return {
        "label": label,
        "score": 0.0,
        "score_type": "heuristic",
        "engine": "fallback",
        "model": None,
    }


def _word_chunks(text: str, max_words: int = 80) -> list[str]:
    words = text.split()
    return [" ".join(words[start:start + max_words]) for start in range(0, len(words), max_words)]


def _load_sentiment_analyzer():
    global _SENTIMENT_ANALYZER, _SENTIMENT_LOAD_ERROR
    if _SENTIMENT_ANALYZER is not None:
        return _SENTIMENT_ANALYZER
    if _SENTIMENT_LOAD_ERROR is not None:
        raise RuntimeError("Pysentimiento indisponível.") from _SENTIMENT_LOAD_ERROR
    try:
        from pysentimiento import create_analyzer

        _SENTIMENT_ANALYZER = create_analyzer(task="sentiment", lang="pt")
        return _SENTIMENT_ANALYZER
    except Exception as error:
        _SENTIMENT_LOAD_ERROR = error
        raise RuntimeError("Pysentimiento indisponível.") from error


def _sentiment_probabilities(output: Any) -> dict[str, float]:
    probabilities = getattr(output, "probas", None)
    if not isinstance(probabilities, dict):
        raise ValueError("Saída inválida do modelo de sentimento.")
    normalized = {str(label).split(".")[-1].upper(): float(score) for label, score in probabilities.items()}
    if not {"POS", "NEG", "NEU"}.issubset(normalized):
        raise ValueError("O modelo de sentimento não devolveu POS, NEG e NEU.")
    return normalized


def _model_sentiment(transcription: str) -> dict[str, Any]:
    analyzer = _load_sentiment_analyzer()
    chunk_probabilities = [
        _sentiment_probabilities(analyzer.predict(chunk))
        for chunk in _word_chunks(transcription)
    ]
    averages = {
        label: sum(item[label] for item in chunk_probabilities) / len(chunk_probabilities)
        for label in ("POS", "NEG", "NEU")
    }
    has_positive = any(item["POS"] >= 0.65 for item in chunk_probabilities)
    has_negative = any(item["NEG"] >= 0.65 for item in chunk_probabilities)
    if has_positive and has_negative:
        label = "misto"
        score = min(averages["POS"] + averages["NEG"], 1.0)
    else:
        winner = max(averages, key=averages.get)
        label = {"POS": "positivo", "NEG": "negativo", "NEU": "neutro"}[winner]
        score = averages[winner]
    return {
        "label": label,
        "score": round(score, 6),
        "score_type": "model_probability",
        "engine": "pysentimiento",
        "model": SENTIMENT_MODEL_NAME,
    }


def _lexical_sentiment(transcription: str) -> dict[str, Any]:
    normalized = f" {_normalize(transcription)} "
    positive_score = 0.0
    negative_score = 0.0
    for phrase, weight in POSITIVE_SENTIMENT.items():
        if f" nao {phrase} " in normalized:
            negative_score += weight
        elif f" {phrase} " in normalized:
            positive_score += weight
    for phrase, weight in NEGATIVE_SENTIMENT.items():
        if f" nao {phrase} " in normalized:
            positive_score += weight
        elif f" {phrase} " in normalized:
            negative_score += weight

    if positive_score and negative_score:
        label = "misto"
    elif positive_score:
        label = "positivo"
    elif negative_score:
        label = "negativo"
    else:
        label = "neutro"
    evidence = positive_score + negative_score
    return {
        "label": label,
        "score": round(min(evidence / 4.0, 1.0), 6),
        "score_type": "heuristic",
        "engine": "lexical_sentiment",
        "model": None,
    }


def _analyze_sentiment(transcription: str, mode: str) -> tuple[dict[str, Any], str, str | None]:
    if mode == "fallback":
        return _lexical_sentiment(transcription), "fallback", None
    try:
        return _model_sentiment(transcription), "model", None
    except Exception as error:
        if mode == "full":
            raise RuntimeError("O modo full exige o modelo Pysentimiento.") from error
        return _lexical_sentiment(transcription), "fallback", type(error).__name__


def _load_churn_classifier():
    global _CHURN_CLASSIFIER, _CHURN_LOAD_ERROR
    if _CHURN_CLASSIFIER is not None:
        return _CHURN_CLASSIFIER
    if _CHURN_LOAD_ERROR is not None:
        raise RuntimeError("MiniLM NLI indisponível.") from _CHURN_LOAD_ERROR
    try:
        from transformers import pipeline

        _CHURN_CLASSIFIER = pipeline("zero-shot-classification", model=CHURN_MODEL_NAME)
        return _CHURN_CLASSIFIER
    except Exception as error:
        _CHURN_LOAD_ERROR = error
        raise RuntimeError("MiniLM NLI indisponível.") from error


def _model_churn(transcription: str) -> dict[str, Any]:
    classifier = _load_churn_classifier()
    descriptions = list(CHURN_HYPOTHESES.values())
    reverse_labels = {description: label for label, description in CHURN_HYPOTHESES.items()}
    scores_by_label = {label: 0.0 for label in CHURN_HYPOTHESES}
    for chunk in _word_chunks(transcription):
        output = classifier(
            chunk,
            candidate_labels=descriptions,
            hypothesis_template="Esta transcrição indica {}.",
            multi_label=False,
        )
        labels = output.get("labels", [])
        scores = output.get("scores", [])
        if len(labels) != len(scores) or not labels:
            raise ValueError("Saída inválida do classificador de churn.")
        for description, score in zip(labels, scores):
            canonical = reverse_labels.get(description)
            if canonical is not None:
                scores_by_label[canonical] = max(scores_by_label[canonical], float(score))
    winner = max(scores_by_label, key=scores_by_label.get)
    return {
        "label": winner,
        "score": round(scores_by_label[winner], 6),
        "score_type": "model_probability",
        "engine": "zero_shot_nli",
        "model": CHURN_MODEL_NAME,
    }


def _lexical_churn(transcription: str) -> dict[str, Any]:
    normalized = f" {_normalize(transcription)} "
    high_score = sum(
        weight for phrase, weight in HIGH_CHURN_SIGNALS.items() if f" {phrase} " in normalized
    )
    medium_score = sum(
        weight for phrase, weight in MEDIUM_CHURN_SIGNALS.items() if f" {phrase} " in normalized
    )
    retention_score = 0.0
    for phrase, weight in RETENTION_SIGNALS.items():
        negated = f" nao {phrase} " in normalized or f" nao vamos {phrase} " in normalized
        if not negated and f" {phrase} " in normalized:
            retention_score += weight

    risk_score = high_score + medium_score
    if high_score >= 3.0 or risk_score >= 4.0:
        label = "alto"
        score = min(0.55 + risk_score / 10.0, 1.0)
    elif risk_score >= 1.5:
        label = "medio"
        score = min(0.35 + risk_score / 10.0, 0.74)
    else:
        label = "baixo"
        score = min(0.3 + retention_score / 10.0, 0.8)
    return {
        "label": label,
        "score": round(score, 6),
        "score_type": "heuristic",
        "engine": "lexical_churn",
        "model": None,
    }


def _analyze_churn(transcription: str, mode: str) -> tuple[dict[str, Any], str, str | None]:
    if mode == "fallback":
        return _lexical_churn(transcription), "fallback", None
    try:
        return _model_churn(transcription), "model", None
    except Exception as error:
        if mode == "full":
            raise RuntimeError("O modo full exige o modelo MiniLM NLI.") from error
        return _lexical_churn(transcription), "fallback", type(error).__name__


def _load_opportunity_classifier():
    global _OPPORTUNITY_CLASSIFIER, _OPPORTUNITY_LOAD_ERROR
    if _OPPORTUNITY_CLASSIFIER is not None:
        return _OPPORTUNITY_CLASSIFIER
    if _OPPORTUNITY_LOAD_ERROR is not None:
        raise RuntimeError("Checkpoint BERTimbau indisponível.") from _OPPORTUNITY_LOAD_ERROR
    try:
        checkpoint = _project_root() / OPPORTUNITY_MODEL_PATH
        if not checkpoint.exists():
            raise FileNotFoundError(f"Checkpoint não encontrado: {checkpoint}")
        from transformers import pipeline

        device = -1
        try:
            import torch

            if torch.cuda.is_available():
                device = 0
        except ImportError:
            pass
        _OPPORTUNITY_CLASSIFIER = pipeline(
            "text-classification",
            model=str(checkpoint),
            tokenizer=str(checkpoint),
            device=device,
        )
        return _OPPORTUNITY_CLASSIFIER
    except Exception as error:
        _OPPORTUNITY_LOAD_ERROR = error
        raise RuntimeError("Checkpoint BERTimbau indisponível.") from error


def _opportunity_probability(output: Any) -> float:
    if isinstance(output, list) and output and isinstance(output[0], list):
        output = output[0]
    if not isinstance(output, list):
        raise ValueError("Saída inválida do classificador de oportunidade.")
    for item in output:
        label = str(item.get("label", "")).casefold()
        if label in {"oportunidade", "label_1", "1"}:
            return float(item["score"])
    raise ValueError("O classificador não devolveu a classe oportunidade.")


def _model_opportunity(transcription: str) -> dict[str, Any]:
    classifier = _load_opportunity_classifier()
    probabilities = [
        _opportunity_probability(
            classifier(chunk, truncation=True, max_length=512, top_k=None)
        )
        for chunk in _word_chunks(transcription, max_words=320)
    ]
    supporting = sum(probability >= 0.80 for probability in probabilities)
    required_support = 1 if len(probabilities) == 1 else 2
    density = supporting / len(probabilities)
    top_probabilities = sorted(probabilities, reverse=True)[:3]
    score = sum(top_probabilities) / len(top_probabilities)
    detected = supporting >= required_support and density >= 0.05
    return {
        "label": "detectada" if detected else "nao_detectada",
        "score": round(score, 6),
        "score_type": "model_probability",
        "engine": "bertimbau_finetuned",
        "model": OPPORTUNITY_MODEL_PATH,
    }


def _signal_hits(normalized: str, signals: set[str]) -> set[str]:
    return {
        signal
        for signal in signals
        if re.search(rf"(?<!\w){re.escape(signal)}(?!\w)", normalized)
    }


def _lexical_opportunity(transcription: str) -> dict[str, Any]:
    normalized = _normalize(transcription)
    refusals = _signal_hits(normalized, OPPORTUNITY_REFUSAL_SIGNALS)
    if refusals:
        return {
            "label": "nao_detectada",
            "score": 1.0,
            "score_type": "heuristic",
            "engine": "lexical_opportunity",
            "model": None,
        }

    intent = _signal_hits(normalized, OPPORTUNITY_INTENT_SIGNALS)
    buying = _signal_hits(normalized, OPPORTUNITY_BUY_SIGNALS)
    pain = _signal_hits(normalized, OPPORTUNITY_PAIN_SIGNALS)
    product = _signal_hits(normalized, OPPORTUNITY_PRODUCT_SIGNALS)
    evidence_score = 2 * bool(intent) + 2 * bool(buying) + bool(pain) + bool(product)
    detected = evidence_score >= 4 and bool(intent or buying)
    return {
        "label": "detectada" if detected else "nao_detectada",
        "score": round(evidence_score / 6.0, 6),
        "score_type": "heuristic",
        "engine": "lexical_opportunity",
        "model": None,
    }


def _analyze_opportunity(transcription: str, mode: str) -> tuple[dict[str, Any], str, str | None]:
    if mode == "fallback":
        return _lexical_opportunity(transcription), "fallback", None
    try:
        return _model_opportunity(transcription), "model", None
    except Exception as error:
        if mode == "full":
            raise RuntimeError("O modo full exige o checkpoint BERTimbau.") from error
        return _lexical_opportunity(transcription), "fallback", type(error).__name__


def _normalize(text: str) -> str:
    normalized = unicodedata.normalize("NFKD", str(text).casefold())
    return "".join(char for char in normalized if not unicodedata.combining(char))


def _tokens(text: str) -> list[str]:
    return [
        token
        for token in re.findall(r"[a-z0-9+]+", _normalize(text))
        if len(token) >= 2 and token not in PORTUGUESE_STOPWORDS
    ]


def _project_root() -> Path:
    configured = globals().get("PROJECT_ROOT")
    if configured is not None:
        root = Path(configured).resolve()
        if (root / "data" / "knowledge_base").exists():
            return root
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "data" / "knowledge_base").exists():
            return candidate
    raise FileNotFoundError("Não foi possível localizar a raiz do projeto Wedjat.")


@lru_cache(maxsize=1)
def _load_catalog() -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    root = _project_root()
    knowledge_base = json.loads(
        (root / "data" / "knowledge_base" / "totvs_rag_kb_v1.json").read_text(
            encoding="utf-8"
        )
    )
    aliases_payload = json.loads(
        (root / "data" / "knowledge_base" / "rag_aliases.json").read_text(
            encoding="utf-8"
        )
    )
    return knowledge_base, aliases_payload["groups"]


def _document_tokens(document: dict[str, Any]) -> list[str]:
    weighted_fields = [
        (document.get("product", ""), 4),
        (document.get("title", ""), 3),
        (" ".join(document.get("keywords", [])), 3),
        (" ".join(document.get("competitors", [])), 2),
        (document.get("category", ""), 1),
        (" ".join(document.get("segments", [])), 1),
        (" ".join(document.get("related_products", [])), 1),
        (document.get("content", ""), 1),
    ]
    return [token for text, weight in weighted_fields for token in _tokens(text) * weight]


def _query_tokens(transcription: str, alias_groups: list[dict[str, Any]]) -> tuple[list[str], set[str]]:
    normalized = f" {_normalize(transcription)} "
    query = _tokens(transcription)
    explicit_products: set[str] = set()
    for group in alias_groups:
        variants = [group["canonical"], *group.get("aliases", [])]
        if any(f" {_normalize(variant)} " in normalized for variant in variants):
            canonical = group["canonical"]
            explicit_products.add(_normalize(canonical))
            query.extend(_tokens(canonical) * 3)
    return query, explicit_products


def _rank_products(transcription: str, top_k: int = 3) -> list[dict[str, Any]]:
    knowledge_base, alias_groups = _load_catalog()
    query, explicit_products = _query_tokens(transcription, alias_groups)
    if not query:
        return []

    document_tokens = [_document_tokens(document) for document in knowledge_base]
    frequencies = [Counter(tokens) for tokens in document_tokens]
    lengths = [len(tokens) for tokens in document_tokens]
    average_length = sum(lengths) / max(len(lengths), 1)
    document_frequency = Counter()
    for tokens in document_tokens:
        document_frequency.update(set(tokens))

    grouped: dict[str, dict[str, Any]] = {}
    for index, (document, tokens, term_frequency, length) in enumerate(
        zip(knowledge_base, document_tokens, frequencies, lengths)
    ):
        score = 0.0
        for term in query:
            frequency = term_frequency.get(term, 0)
            if not frequency:
                continue
            seen_in = document_frequency[term]
            inverse = math.log(1 + (len(knowledge_base) - seen_in + 0.5) / (seen_in + 0.5))
            denominator = frequency + 1.5 * (1 - 0.75 + 0.75 * length / average_length)
            score += inverse * frequency * 2.5 / denominator

        product = document.get("product") or document.get("title")
        normalized_product = _normalize(product)
        if any(alias in normalized_product or normalized_product in alias for alias in explicit_products):
            score += 8.0
        if score < 2.0:
            continue

        source_urls = [source["url"] for source in document.get("sources", []) if source.get("url")]
        if not source_urls:
            continue
        matched = set(query) & set(tokens)
        candidate = grouped.setdefault(
            product,
            {"raw_score": 0.0, "matched_terms": set(), "document_ids": [], "sources": []},
        )
        candidate["raw_score"] = max(candidate["raw_score"], score)
        candidate["matched_terms"].update(matched)
        candidate["document_ids"].append(document["id"])
        candidate["sources"].extend(source_urls)

    ranked = sorted(grouped.items(), key=lambda item: (-item[1]["raw_score"], item[0]))[:top_k]
    if not ranked:
        return []
    highest_score = ranked[0][1]["raw_score"]
    return [
        {
            "product": product,
            "score": round(values["raw_score"] / highest_score, 6),
            "score_type": "heuristic",
            "engine": "bm25_aliases",
            "model": None,
            "matched_terms": sorted(values["matched_terms"])[:10],
            "document_ids": list(dict.fromkeys(values["document_ids"])),
            "sources": list(dict.fromkeys(values["sources"])),
        }
        for product, values in ranked
    ]


def analisar_transcricao(transcricao: str, modo: str = "auto") -> dict[str, Any]:
    """Analisa uma transcrição e devolve o contrato comercial da Sprint 4."""
    if not isinstance(transcricao, str):
        raise TypeError("A transcrição deve ser uma string.")
    if not transcricao.strip():
        raise ValueError("A transcrição não pode estar vazia.")
    if modo not in SUPPORTED_MODES:
        validos = ", ".join(sorted(SUPPORTED_MODES))
        raise ValueError(f"Modo inválido: {modo!r}. Use um de: {validos}.")

    produtos_candidatos = _rank_products(transcricao)
    sentimento, sentiment_mode, sentiment_reason = _analyze_sentiment(transcricao, modo)
    risco_churn, churn_mode, churn_reason = _analyze_churn(transcricao, modo)
    oportunidade, opportunity_mode, opportunity_reason = _analyze_opportunity(
        transcricao, modo
    )
    fallback_reasons = {}
    if sentiment_reason:
        fallback_reasons["sentiment"] = sentiment_reason
    if churn_reason:
        fallback_reasons["churn"] = churn_reason
    if opportunity_reason:
        fallback_reasons["opportunity"] = opportunity_reason

    return {
        "schema_version": "1.0",
        "transcricao_original": transcricao,
        "produto_identificado": (
            produtos_candidatos[0]["product"] if produtos_candidatos else None
        ),
        "produtos_candidatos": produtos_candidatos,
        "sentimento": sentimento,
        "risco_churn": risco_churn,
        "oportunidade_comercial": oportunidade,
        "principais_termos": [],
        "recomendacao_acao": {
            "label": "acompanhar_conta",
            "revisao_humana": True,
        },
        "analysis_mode": {
            "requested": modo,
            "components": {
                "products": "fallback",
                "sentiment": sentiment_mode,
                "churn": churn_mode,
                "opportunity": opportunity_mode,
            },
            "fallback_reasons": fallback_reasons,
        },
    }


## 5. Nova transcrição

Substitua o conteúdo de `TRANSCRICAO` pelo texto original da reunião. Apenas uma transcrição é processada por execução.

In [ ]:
TRANSCRICAO = """
Cole aqui a transcrição original da reunião.
""".strip()

MODO_ANALISE = "auto"  # auto, full ou fallback

## 6. Executar e revisar

A revisão humana é obrigatória. Produto, sentimento, churn e oportunidade já estão integrados; termos principais e recomendação serão acrescentados nas próximas etapas.

In [ ]:
import json

resultado = analisar_transcricao(TRANSCRICAO, modo=MODO_ANALISE)
print(json.dumps(resultado, ensure_ascii=False, indent=2))